In [ ]:
import geopandas as gpd
import requests
from bs4 import BeautifulSoup
import re

In [ ]:
# loading Shape
shapefile_path = "vw_btpolygonproved.shp"
gdf = gpd.read_file(shapefile_path)

In [ ]:
gdf.shape

In [ ]:
# New Column creation
gdf['tc_code'] = None
gdf['tc_name'] = None

In [ ]:
def extract_info_from_web(url):
    try:
        # Fetching the Web Page Content
        response = requests.get(url)
        soup =あ BeautifulSoup(response.content, 'html.parser')

        pattern = re.compile(r'(.*)\n') # Compiling a Regular Expression Pattern
        text = soup.find("li").text # Extracting Text from HTML
        match= pattern.match(text) # Matching the Text Against the Regular Expression
        code_and_name = match.group(0)
        # print(code_and_name)
        pattern = re.compile(r'([^-]*)-(.*)\n') # Compiling Another Regular Expression Pattern
        match= pattern.match(code_and_name) # Matching the Extracted Text Against the New Pattern
        code = match.group(1).strip()
        name = match.group(2).strip()
        return code, name
    except Exception as e:
        print(f"Error fetching data from {url}: {e}")
        return None, None

In [ ]:
# Retrieve information for each record using a web link and add it to the column
for index, row in gdf.iterrows():
    web_url = row['LINK'] 
    column1_info, column2_info = extract_info_from_web(web_url)
    gdf.at[index, 'tc_code'] = column1_info
    gdf.at[index, 'tc_name'] = column2_info

In [ ]:
# Save updated Shapefile
gdf.to_file('vm_btpolygonproved_new.shp')